# Build Two-Constraint Instruction Triplets

This notebook combines each seed instruction with the corresponding `org_constraint` and `new_constraint` from `datasets/conflict_instruction.jsonl`.

For each available conflict type, it creates:

- `original_instruction`: seed + original constraint
- `new_instruction`: seed + new constraint
- `conflict_instruction`: seed + original constraint + new constraint

No language model is required.

In [1]:
import json
from collections import Counter
from pathlib import Path


def find_repo_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "datasets" / "seed_instruction.jsonl").exists():
            return candidate
    raise FileNotFoundError("Could not find the repository root containing datasets/.")


REPO_ROOT = find_repo_root()
SEED_FILE = REPO_ROOT / "datasets" / "seed_instruction.jsonl"
CONFLICT_FILE = REPO_ROOT / "datasets" / "conflict_instruction.jsonl"
OUTPUT_FILE = REPO_ROOT / "datasets" / "constraint_triplets.jsonl"

print(f"Repository root: {REPO_ROOT}")
print(f"Output file: {OUTPUT_FILE}")

Repository root: /Users/xinmingwang/Library/CloudStorage/OneDrive-NationalUniversityofSingapore/ConInstruct-master
Output file: /Users/xinmingwang/Library/CloudStorage/OneDrive-NationalUniversityofSingapore/ConInstruct-master/datasets/constraint_triplets.jsonl


In [2]:
CONFLICT_TYPES = {
    1: "Conflicts between Content Constraints",
    2: "Conflicts between Keyword Constraints",
    3: "Conflicts between Keyword Constraints and Phrase Constraints",
    4: "Conflicts between Phrase Constraints",
    5: "Conflicts between Length Constraints",
    6: "Conflicts between Format Constraints",
    7: "Conflicts between Style Constraints",
    8: "Conflicts between Phrase Constraints and Content Constraints",
    9: "Conflicts between Phrase Constraints and Style Constraints",
}


def read_jsonl(path):
    with path.open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


seed_rows = read_jsonl(SEED_FILE)
conflict_rows = read_jsonl(CONFLICT_FILE)

if len(seed_rows) != len(conflict_rows):
    raise ValueError(
        f"The files must align by row, but found {len(seed_rows)} seed rows "
        f"and {len(conflict_rows)} conflict rows."
    )

print(f"Loaded {len(seed_rows)} aligned samples.")

Loaded 100 aligned samples.


In [3]:
def append_parts(*parts):
    return " ".join(part.strip() for part in parts if part and part.strip())


triplets = []
skipped = []

for sample_id, (seed_row, conflict_row) in enumerate(
    zip(seed_rows, conflict_rows), start=1
):
    seed_instruction = seed_row["instruction"].strip()

    for conflict_type_idx, conflict_type in CONFLICT_TYPES.items():
        conflict_info = conflict_row.get(conflict_type)
        if not conflict_info:
            skipped.append((sample_id, conflict_type_idx, "missing conflict type"))
            continue

        org_constraint = conflict_info.get("org_constraint", "").strip()
        new_constraint = conflict_info.get("new_constraint", "").strip()
        if not org_constraint or not new_constraint:
            skipped.append((sample_id, conflict_type_idx, "missing constraint text"))
            continue

        triplets.append(
            {
                "sample_id": sample_id,
                "task": seed_row.get("task"),
                "conflict_type_idx": conflict_type_idx,
                "conflict_type": conflict_type,
                "seed_instruction": seed_instruction,
                "org_constraint": org_constraint,
                "new_constraint": new_constraint,
                "original_instruction": append_parts(
                    seed_instruction, org_constraint
                ),
                "new_instruction": append_parts(
                    seed_instruction, new_constraint
                ),
                "conflict_instruction": append_parts(
                    seed_instruction, org_constraint, new_constraint
                ),
            }
        )

print(f"Built {len(triplets)} triplets.")
print(f"Skipped {len(skipped)} unavailable sample/type pairs.")

Built 864 triplets.
Skipped 36 unavailable sample/type pairs.


In [4]:
with OUTPUT_FILE.open("w", encoding="utf-8") as file:
    for item in triplets:
        file.write(json.dumps(item, ensure_ascii=False) + "\n")

counts = Counter(item["conflict_type_idx"] for item in triplets)
print(f"Wrote {len(triplets)} records to {OUTPUT_FILE}")
for conflict_type_idx, conflict_type in CONFLICT_TYPES.items():
    print(f"Type {conflict_type_idx}: {counts[conflict_type_idx]:3d} | {conflict_type}")

Wrote 864 records to /Users/xinmingwang/Library/CloudStorage/OneDrive-NationalUniversityofSingapore/ConInstruct-master/datasets/constraint_triplets.jsonl
Type 1: 100 | Conflicts between Content Constraints
Type 2: 100 | Conflicts between Keyword Constraints
Type 3: 100 | Conflicts between Keyword Constraints and Phrase Constraints
Type 4: 100 | Conflicts between Phrase Constraints
Type 5: 100 | Conflicts between Length Constraints
Type 6: 100 | Conflicts between Format Constraints
Type 7: 100 | Conflicts between Style Constraints
Type 8:  94 | Conflicts between Phrase Constraints and Content Constraints
Type 9:  70 | Conflicts between Phrase Constraints and Style Constraints


In [5]:
preview = triplets[0]
for key in (
    "sample_id",
    "conflict_type_idx",
    "original_instruction",
    "new_instruction",
    "conflict_instruction",
):
    print(f"\n{key}:\n{preview[key]}")


sample_id:
1

conflict_type_idx:
1

original_instruction:
Write an email to my boss telling him that I am quitting. The email must include specific reasons for my resignation, such as career growth opportunities or personal circumstances.

new_instruction:
Write an email to my boss telling him that I am quitting. Do not include any reasons for your resignation in the email.

conflict_instruction:
Write an email to my boss telling him that I am quitting. The email must include specific reasons for my resignation, such as career growth opportunities or personal circumstances. Do not include any reasons for your resignation in the email.


In [1]:
print("thought\nWedding planning timeline for a couple getting married in six months.\nMonthly tasks.\nThird person (e.g., \"The couple,\" \"They,\" \"The bride and groom\").\nInclude the phrase \"We are excited to plan this wedding together.\"\n\n    *   *Month 6 (6 Months Out):* Foundation (Venue, Guest List, Budget, Theme).\n    *   *Month 5 (5 Months Out):* Vendors (Catering, Photographer, Planner, Officiant).\n    *   *Month 4 (4 Months Out):* Attire & Decor (Dress/Suit, Invitations, Decor details).\n    *   *Month 3 (3 Months Out):* Logistics (Cake, Music, Rehearsal, Registry).\n    *   *Month 2 (2 Months Out):* Finalizing (RSVPs, Fittings, Menu tasting).\n    *   *Month 1 (1 Month Out):* Final details (Seating chart, Final counts, Packing).\n\n    *   *Introduction:* Set the scene. Mention the six-month window.\n    *   *Month 6:* The couple establishes the budget. They book the venue. They draft the guest list.\n    *   *Month 5:* They hire the photographer and caterer. They choose the wedding party.\n    *   *Month 4:* They order the attire. They send out Save the Dates. They finalize the color palette.\n    *   *Month 3:* They order the invitations. They book the florist and baker. They plan the honeymoon.\n    *   *Month 2:* They track RSVPs. They do dress fittings. They finalize the music playlist.\n    *   *Month 1:* They create the seating chart. They confirm final headcounts. They prepare for the big day.\n\n    *   *Third person?* Yes.\n    *   *Six months?* Yes.\n    *   *Monthly tasks?* Yes.\n    *   *Required phrase?* \"We are excited to plan this wedding together.\" (Need to place this naturally, perhaps as a concluding note or a transition).\n\n    *   *Month 6:* The couple begins by establishing a comprehensive budget and a preliminary guest list. They secure their primary venue and select a wedding theme.\n    *   *Month 5:* They focus on booking key vendors, including the caterer, photographer, and videographer. They also begin searching for the wedding party.\n    *   *Month 4:* The couple selects their wedding attire and begins the process of ordering invitations. They finalize the floral arrangements and decor style.\n    *   *Month 3:* They send out the formal invitations and book the wedding cake. They also begin planning the rehearsal dinner and the honeymoon itinerary.\n    *   *Month 2:* The couple manages the RSVP list and conducts final dress and suit fittings. They meet with the caterer for a menu tasting and finalize the music selection.\n    *   *Month 1:* They complete the seating chart and provide final headcounts to all vendors. They assemble wedding day emergency kits and prepare for the ceremony.\n\n    *   *Wait, the prompt says \"The response must include the phrase...\"* I should make sure it's clearly visible.\n\n    *   *Refining the third person:* Ensure no \"I\" or \"You\" except in the required phrase.\n\n    *   *Drafting final response:*\n    \"With only six months remaining until the big day, the couple must move quickly to ensure every detail is perfect. We are excited to plan this wedding together.\n\n    **Month 6: The Foundation**\n    The couple establishes a firm budget and creates a preliminary guest list. They secure their wedding venue and select a general theme or color palette to guide future decisions.\n\n    **Month 5: Securing the Team**\n    They focus on booking essential vendors, such as the caterer, photographer, and videographer. During this month, they also officially ask their wedding party members to participate.\n\n    **Month 4: Style and Stationery**\n    The couple selects their wedding attire and begins the process of ordering invitations. They finalize the floral designs and begin coordinating with the decorator for specific aesthetic details.\n\n    **Month 3: Logistics and Details**\n    They send out the formal invitations and book the wedding cake. The couple also begins planning the rehearsal dinner and starts organizing their honeymoon itinerary.\n\n    **Month 2: Finalizing the Experience**\n    The couple manages the RSVP list and attends final dress and suit fittings. They conduct a menu tasting with the caterer and finalize the music playlist for the ceremony and reception.\n\n    **Month 1: The Final Countdown**\n    They complete the seating chart and provide final headcounts to all vendors. The couple assembles wedding day emergency kits, confirms the timeline with the officiant, and prepares for the celebration.\"\n\n    *   Third person? Yes.\n    *   Six months? Yes.\n    *   Monthly tasks? Yes.\n    *   Required phrase? Yes.With only six months remaining until the big day, the couple must move efficiently to ensure every detail is perfect. We are excited to plan this wedding together.\n\n**Month 6: The Foundation**\nThe couple begins by establishing a firm budget and creating a preliminary guest list. They secure their primary wedding venue and select a general theme or color palette to guide all future aesthetic decisions.\n\n**Month 5: Securing the Team**\nThey focus on booking essential vendors, including the caterer, photographer, and videographer. During this month, they also officially ask their wedding party members to participate and begin researching officiants.\n\n**Month 4: Style and Stationery**\nThe couple selects their wedding attire and begins the process of ordering invitations. They finalize the floral designs and begin coordinating with the decorator to ensure the venue's atmosphere aligns with their vision.\n\n**Month 3: Logistics and Details**\nThey send out the formal invitations and book the wedding cake. The couple also begins planning the rehearsal dinner and starts organizing their honeymoon itinerary and travel arrangements.\n\n**Month 2: Finalizing the Experience**\nThe couple manages the RSVP list and attends final dress and suit fittings. They conduct a menu tasting with the caterer, finalize the music playlist for the ceremony and reception, and confirm the timeline with all primary vendors.\n\n**Month 1: The Final Countdown**\nThey complete the seating chart and provide final headcounts to the venue and caterer. The couple assembles wedding day emergency kits, confirms the final schedule with the wedding party, and prepares for the celebration.")

thought
Wedding planning timeline for a couple getting married in six months.
Monthly tasks.
Third person (e.g., "The couple," "They," "The bride and groom").
Include the phrase "We are excited to plan this wedding together."

    *   *Month 6 (6 Months Out):* Foundation (Venue, Guest List, Budget, Theme).
    *   *Month 5 (5 Months Out):* Vendors (Catering, Photographer, Planner, Officiant).
    *   *Month 4 (4 Months Out):* Attire & Decor (Dress/Suit, Invitations, Decor details).
    *   *Month 3 (3 Months Out):* Logistics (Cake, Music, Rehearsal, Registry).
    *   *Month 2 (2 Months Out):* Finalizing (RSVPs, Fittings, Menu tasting).
    *   *Month 1 (1 Month Out):* Final details (Seating chart, Final counts, Packing).

    *   *Introduction:* Set the scene. Mention the six-month window.
    *   *Month 6:* The couple establishes the budget. They book the venue. They draft the guest list.
    *   *Month 5:* They hire the photographer and caterer. They choose the wedding party.
    * 